# Oscillatory modes — v2 checkpoint

Same analyses as `05_oscillations.ipynb`, but on the **v2 model** (`sigma_rec=0.10`,
`noise_at_eval=True`). Additionally includes **jPCA with pca_dims=3** to connect
oscillatory structure to rotational dynamics (matching Amengual et al. methodology).

## Analyses
1. Eigenspectrum of W_rec_eff (structural)
2. Jacobian eigenspectrum at mean pre-target state (functional)
3. Welch PSD of delay-period activity
4. PSD by CTOA bin
5. jPCA (pca_dims=3) — rotational frequency from M_skew eigenvalues
6. Comparison: Jacobian freq vs PSD peak vs jPCA freq

In [ ]:
import sys

sys.path.insert(0, "..")

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy import signal as sp_signal

from src.model import BioLeakyRNN
from src.env import CuedTargetWithDistractorsV3
from src.analysis import (
    collect_trials,
    prepare_jpca_input,
    fit_jpca,
)

device = "cpu"
print("device:", device)

## 1. Load v2 model

In [ ]:
from pathlib import Path

ckpt_v2 = Path("../checkpoints/stage2_v2.pt")
ckpt_v1 = Path("../checkpoints/stage2.pt")

if ckpt_v2.exists():
    ckpt_path = ckpt_v2
    sigma_rec = 0.10
    print("Using v2 checkpoint")
else:
    ckpt_path = ckpt_v1
    sigma_rec = 0.05
    print("v2 not found, using v1")


def make_model():
    return BioLeakyRNN(
        input_size=7,
        hidden_size=128,
        output_size=2,
        dt=20.0,
        tau=100.0,
        activation="softplus",
        sigma_rec=sigma_rec,
        use_ei=True,
        exc_ratio=0.7,
        use_dale=True,
        mask_seed=42,
    )


def make_env():
    return CuedTargetWithDistractorsV3(
        dt=20, cue_strength=1.0, p_distractor_trial=0.6, distractor_strength=1.0
    )


model = make_model().to(device)
model.load_state_dict(torch.load(str(ckpt_path), weights_only=True)["state_dict"])
model.eval()
model.noise_at_eval = True
print(f"Loaded {ckpt_path.name}, noise_at_eval={model.noise_at_eval}")

alpha = float(model.alpha)  # dt/tau = 20/100 = 0.2
dt_ms = model.dt  # 20 ms
print(f"alpha={alpha:.4f},  dt={dt_ms} ms,  tau={model.tau} ms")

## 2. Eigenspectrum of W_rec_eff (structural analysis)

Effective recurrent weight matrix with Dale's law:
`W_rec_eff[j,i] = |W_rec[j,i]| * ei_sign[i]`

In [ ]:
W_eff = model.effective_W_rec().detach().cpu().numpy()  # [128, 128]
print(f"W_rec_eff shape: {W_eff.shape}")
print(f"E neurons: {model.n_exc},  I neurons: {model.n_inh}")
print(f"Spectral radius (max |lam|): {np.abs(np.linalg.eigvals(W_eff)).max():.4f}")

eigvals_W = np.linalg.eigvals(W_eff)

theta_W = np.angle(eigvals_W)
freq_W = np.abs(theta_W) / (2 * np.pi * dt_ms / 1000)  # Hz

print(f"\nTop-10 W_rec eigenvalues by |Im|:")
idx = np.argsort(-np.abs(eigvals_W.imag))
for i in idx[:10]:
    lam = eigvals_W[i]
    print(
        f"  |lam|={np.abs(lam):.3f}  Re={lam.real:+.3f}  Im={lam.imag:+.3f}  "
        f"freq={freq_W[i]:.2f} Hz"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: eigenspectrum on complex plane
ax = axes[0]
th = np.linspace(0, 2 * np.pi, 300)
ax.plot(np.cos(th), np.sin(th), "--", color="gray", alpha=0.5, lw=1, label="unit circle")
sc = ax.scatter(
    eigvals_W.real,
    eigvals_W.imag,
    c=np.abs(eigvals_W.imag),
    cmap="plasma",
    s=20,
    alpha=0.8,
)
plt.colorbar(sc, ax=ax, label="|Im(lam)|")
ax.axhline(0, color="gray", lw=0.5)
ax.axvline(0, color="gray", lw=0.5)
ax.set_xlabel("Re(lam)")
ax.set_ylabel("Im(lam)")
ax.set_title("Eigenspectrum of W_rec_eff (v2)")
ax.axis("equal")
ax.legend(fontsize=8)

# Right: histogram of implied frequencies
ax2 = axes[1]
osc_mask = np.abs(eigvals_W.imag) > 0.05
ax2.hist(freq_W[osc_mask], bins=30, color="steelblue", alpha=0.8)
ax2.set_xlabel("Implied frequency (Hz)")
ax2.set_ylabel("count")
ax2.set_title("Distribution of oscillatory frequencies\nin W_rec_eff (v2)")

plt.tight_layout()
plt.show()

## 3. Jacobian eigenspectrum (functional analysis)

Actual dynamics Jacobian at the mean pre-target state:
```
A = (1 - alpha) I + alpha * diag(phi'(h*)) * W_rec_eff
```

In [ ]:
print("Collecting 5000 trials (with noise) for pre-target state estimation...")
trials = collect_trials(model, make_env, n_trials=5000, device=device)
correct = [tr for tr in trials if tr["train_outcome"] == "correct"]
print(f"Correct trials: {len(correct)}")

# Mean hidden state in pre-target window (-300..-100 ms = steps -15..-5)
pre_states = []
for tr in correct:
    t0 = tr["target_onset"]
    s, e = t0 - 15, t0 - 5
    if s >= 0 and e <= tr["h"].shape[0]:
        pre_states.append(tr["h"][s:e].mean(axis=0))

h_star = np.mean(pre_states, axis=0)  # [128]
print(
    f"Mean pre-target state: min={h_star.min():.3f}, max={h_star.max():.3f}, "
    f"mean={h_star.mean():.3f}"
)

In [ ]:
# Softplus derivative: phi'(x) = sigmoid(x)
h_star_t = torch.tensor(h_star, dtype=torch.float32)
preact_star = torch.log(torch.expm1(h_star_t.clamp(min=1e-6)))
dphi = torch.sigmoid(preact_star).numpy()  # [128]

# Build Jacobian A = (1-alpha)*I + alpha * diag(dphi) @ W_eff
I = np.eye(128)
A = (1 - alpha) * I + alpha * (dphi[:, None] * W_eff)

eigvals_A = np.linalg.eigvals(A)
theta_A = np.angle(eigvals_A)
freq_A = np.abs(theta_A) / (2 * np.pi * dt_ms / 1000)  # Hz
modulus_A = np.abs(eigvals_A)

print(f"Spectral radius of A: {modulus_A.max():.4f}")
print(f"(>1 = unstable, ~1 = sustained, <1 = damped)")
print()
print("Top-10 Jacobian eigenvalues by |Im|:")
idx = np.argsort(-np.abs(eigvals_A.imag))
for i in idx[:10]:
    lam = eigvals_A[i]
    f = freq_A[i]
    if f > 0:
        print(
            f"  |lam|={modulus_A[i]:.4f}  Re={lam.real:+.4f}  Im={lam.imag:+.4f}  "
            f"freq={f:.2f} Hz  period={1000/f:.0f} ms"
        )
    else:
        print(
            f"  |lam|={modulus_A[i]:.4f}  Re={lam.real:+.4f}  Im={lam.imag:+.4f}  "
            f"freq=0 Hz"
        )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Jacobian eigenspectrum
ax = axes[0]
th = np.linspace(0, 2 * np.pi, 300)
ax.plot(np.cos(th), np.sin(th), "--", color="gray", alpha=0.5, lw=1, label="unit circle")
sc = ax.scatter(
    eigvals_A.real,
    eigvals_A.imag,
    c=np.abs(eigvals_A.imag),
    cmap="plasma",
    s=25,
    alpha=0.85,
)
plt.colorbar(sc, ax=ax, label="|Im(lam)|")
ax.axhline(0, color="gray", lw=0.5)
ax.axvline(0, color="gray", lw=0.5)
ax.set_xlabel("Re(lam)")
ax.set_ylabel("Im(lam)")
ax.set_title("Jacobian eigenspectrum A (v2)\n(linearised at mean pre-target state)")
ax.axis("equal")
ax.legend(fontsize=8)

# Right: |lam| vs frequency
ax2 = axes[1]
osc_mask = np.abs(eigvals_A.imag) > 0.001
ax2.scatter(
    freq_A[osc_mask],
    modulus_A[osc_mask],
    c=modulus_A[osc_mask],
    cmap="RdYlGn",
    s=30,
    alpha=0.85,
    vmin=0.7,
    vmax=1.1,
)
ax2.axhline(1.0, color="red", ls="--", lw=1, alpha=0.6, label="|lam|=1 (sustained)")
ax2.set_xlabel("Frequency (Hz)")
ax2.set_ylabel("|lam| (modulus)")
ax2.set_title("Oscillatory modes: frequency vs modulus")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4. Power spectrum of pre-target hidden activity

Welch PSD of delay-period neural activity (cue onset to target - 10 steps).

In [ ]:
# Extract delay-period activity
delay_segs = []
for tr in correct:
    cue = tr["cue_onset"]
    tgt = tr["target_onset"]
    seg_end = tgt - 10  # avoid pre-target ramp
    if seg_end - cue >= 10:  # at least 200 ms
        delay_segs.append(tr["h"][cue:seg_end])

print(f"Delay segments: {len(delay_segs)}")
lengths = [s.shape[0] for s in delay_segs]
print(
    f"Delay length: min={min(lengths)}, max={max(lengths)}, "
    f"mean={np.mean(lengths):.1f} steps  ({np.mean(lengths)*dt_ms:.0f} ms)"
)

In [ ]:
fs = 1000 / dt_ms  # 50 Hz

all_psds = []
for seg in delay_segs:
    T, N = seg.shape
    seg_c = seg - seg.mean(axis=0, keepdims=True)
    freqs, psd = sp_signal.welch(
        seg_c.T, fs=fs, nperseg=min(T, 16), noverlap=None, scaling="density"
    )
    all_psds.append(psd.mean(axis=0))

psd_mean = np.mean(all_psds, axis=0)
psd_sem = np.std(all_psds, axis=0) / np.sqrt(len(all_psds))

print(f"Frequency resolution: {freqs[1]-freqs[0]:.2f} Hz")
print(f"Max frequency: {freqs[-1]:.1f} Hz  (Nyquist = {fs/2:.1f} Hz)")

peak_idx = np.argmax(psd_mean[1:]) + 1
print(
    f"\nDominant peak: {freqs[peak_idx]:.2f} Hz  "
    f"(period = {1000/freqs[peak_idx]:.0f} ms)"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: mean power spectrum
ax = axes[0]
ax.semilogy(freqs[1:], psd_mean[1:], color="steelblue", lw=2, label="mean PSD")
ax.fill_between(
    freqs[1:],
    psd_mean[1:] - psd_sem[1:],
    psd_mean[1:] + psd_sem[1:],
    alpha=0.25,
    color="steelblue",
)
ax.axvline(
    freqs[peak_idx], color="red", ls="--", lw=1.5, label=f"peak: {freqs[peak_idx]:.1f} Hz"
)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power spectral density")
ax.set_title("Pre-target delay PSD (v2, noise at eval)")
ax.legend(fontsize=9)

# Right: per-neuron heatmap
ax2 = axes[1]
ex_idx = np.argmax(lengths)
seg_ex = delay_segs[ex_idx]
seg_ex_c = seg_ex - seg_ex.mean(axis=0, keepdims=True)
_, psd_ex = sp_signal.welch(
    seg_ex_c.T, fs=fs, nperseg=min(seg_ex.shape[0], 16), noverlap=None, scaling="density"
)
peak_power = psd_ex[:, peak_idx]
sort_idx = np.argsort(-peak_power)
im = ax2.imshow(
    np.log10(psd_ex[sort_idx, 1:] + 1e-12),
    aspect="auto",
    origin="lower",
    extent=[freqs[1], freqs[-1], 0, 128],
    cmap="viridis",
)
plt.colorbar(im, ax=ax2, label="log10(PSD)")
ax2.axvline(freqs[peak_idx], color="red", ls="--", lw=1.5)
ax2.set_xlabel("Frequency (Hz)")
ax2.set_ylabel("Neuron (sorted by peak power)")
ax2.set_title(
    f"PSD per neuron (longest trial)\nsorted by power at {freqs[peak_idx]:.1f} Hz"
)

plt.tight_layout()
plt.show()

## 5. PSD by CTOA bin

Do longer delays show different oscillatory patterns?

In [ ]:
groups = {"early (bins 0-2)": (0, 2), "mid (bins 4-6)": (4, 6), "late (bins 7-9)": (7, 9)}
colors = ["steelblue", "orange", "crimson"]

fig, ax = plt.subplots(figsize=(9, 5))

for (label, (bmin, bmax)), col in zip(groups.items(), colors):
    group_psds = []
    for tr in correct:
        b = tr.get("ctoa_bin")
        if b is None or not (bmin <= b <= bmax):
            continue
        cue = tr["cue_onset"]
        tgt = tr["target_onset"]
        seg_end = tgt - 10
        if seg_end - cue < 10:
            continue
        seg = tr["h"][cue:seg_end]
        seg_c = seg - seg.mean(axis=0, keepdims=True)
        _, psd = sp_signal.welch(
            seg_c.T,
            fs=fs,
            nperseg=min(seg.shape[0], 16),
            noverlap=None,
            scaling="density",
        )
        group_psds.append(psd.mean(axis=0))

    if not group_psds:
        continue
    gm = np.mean(group_psds, axis=0)
    gs = np.std(group_psds, axis=0) / np.sqrt(len(group_psds))
    ax.semilogy(
        freqs[1:], gm[1:], color=col, lw=2, label=f"{label} (n={len(group_psds)})"
    )
    ax.fill_between(freqs[1:], gm[1:] - gs[1:], gm[1:] + gs[1:], alpha=0.15, color=col)

ax.axvline(
    freqs[peak_idx], color="gray", ls="--", lw=1, label=f"peak {freqs[peak_idx]:.1f} Hz"
)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power spectral density")
ax.set_title("Pre-target delay PSD by CTOA bin (v2)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 6. jPCA rotational dynamics (pca_dims=3)

Connect oscillation analysis to jPCA: the eigenvalues of M_skew give
the rotational frequency in PCA space. Compare with Jacobian and PSD.

In [ ]:
X_cond, labels, rel_time, counts = prepare_jpca_input(
    trials,
    align_key="target_onset",
    window_before=15,
    window_after=30,
    min_trials=10,
    outcome="correct",
)
print(f"X_cond shape: {X_cond.shape}")
print(f"Trials per bin: {counts}")

# Fit jPCA with pca_dims=3 (Amengual paper) and pca_dims=6 (Churchland)
for dims in [3, 6]:
    res = fit_jpca(X_cond, n_jpcs=1, pca_dims=dims)

    eigs = res["eigenvalues"]
    # jPCA rotation frequency: M_skew eigenvalues are in units of 1/step
    # M_skew describes dX/dt ~ M_skew @ X, so omega = |Im(eig)| per step
    # freq = omega / (2*pi) * (1000/dt)  [Hz]
    top_omega = np.max(np.abs(eigs.imag))  # radians per step
    jpca_freq = top_omega / (2 * np.pi) * (1000 / dt_ms)  # Hz
    jpca_period = 1000 / jpca_freq if jpca_freq > 0 else float("inf")

    evr = res["pca_pre"].explained_variance_ratio_
    X_flat = X_cond.reshape(-1, X_cond.shape[-1])
    total_var = np.var(X_flat, axis=0).sum()
    Z_flat = res["Z"].reshape(-1, res["Z"].shape[-1])
    jpc_var = np.var(Z_flat, axis=0)
    jpc12_pct = jpc_var[:2].sum() / total_var * 100

    print(f"\n=== jPCA pca_dims={dims} ===")
    print(f"PCA variance: {evr.sum()*100:.1f}%")
    print(f'var_ratio (raw-dX):     {res["var_ratio"]:.3f}')
    print(f'var_ratio (Churchland): {res["var_ratio_churchland"]:.3f}')
    print(f'rot_index:              {res["rot_index"]:.3f}')
    print(f"jPC1+2:                 {jpc12_pct:.1f}%")
    print(f"Top M_skew eigenvalue:  {eigs[0].imag:+.4f}i")
    print(f"Rotational frequency:   {jpca_freq:.2f} Hz")
    print(f"Rotational period:      {jpca_period:.0f} ms")

## 7. Summary: cross-method frequency comparison

Comparing oscillation frequencies from three independent analyses.

In [ ]:
# W_rec: top oscillatory eigenvalue
idx_W = np.argsort(-np.abs(eigvals_W.imag))
w_top = eigvals_W[idx_W[0]]
w_freq = freq_W[idx_W[0]]

# Jacobian: top oscillatory eigenvalue
idx_A = np.argsort(-np.abs(eigvals_A.imag))
a_top = eigvals_A[idx_A[0]]
a_freq = freq_A[idx_A[0]]

# PSD peak
psd_freq = freqs[peak_idx]

# jPCA (pca_dims=3)
res3 = fit_jpca(X_cond, n_jpcs=1, pca_dims=3)
jpca3_omega = np.max(np.abs(res3["eigenvalues"].imag))
jpca3_freq = jpca3_omega / (2 * np.pi) * (1000 / dt_ms)

# jPCA (pca_dims=6)
res6 = fit_jpca(X_cond, n_jpcs=1, pca_dims=6)
jpca6_omega = np.max(np.abs(res6["eigenvalues"].imag))
jpca6_freq = jpca6_omega / (2 * np.pi) * (1000 / dt_ms)

print("=" * 65)
print(f'{"Method":<30} {"Frequency (Hz)":>15} {"Period (ms)":>15}')
print("-" * 65)
print(f'{"W_rec top eigenvalue":<30} {w_freq:>15.2f} {1000/w_freq:>15.0f}')
print(f'{"Jacobian top eigenvalue":<30} {a_freq:>15.2f} {1000/a_freq:>15.0f}')
print(f'{"Welch PSD peak":<30} {psd_freq:>15.2f} {1000/psd_freq:>15.0f}')
print(f'{"jPCA M_skew (pca_dims=3)":<30} {jpca3_freq:>15.2f} {1000/jpca3_freq:>15.0f}')
print(f'{"jPCA M_skew (pca_dims=6)":<30} {jpca6_freq:>15.2f} {1000/jpca6_freq:>15.0f}')
print("=" * 65)

## 8. Comparison: v1 vs v2

Compare key oscillation metrics between v1 and v2 checkpoints.

In [ ]:
print("=" * 65)
print(f'{"Metric":<35} {"v1 (05_osc)":>13} {"v2 (this)":>13}')
print("-" * 65)
print(f'{"W_rec spectral radius":<35} {"3.1519":>13} {np.abs(eigvals_W).max():>13.4f}')
print(f'{"W_rec top freq (Hz)":<35} {"10.05":>13} {w_freq:>13.2f}')
print(f'{"Jacobian spectral radius":<35} {"1.0003":>13} {modulus_A.max():>13.4f}')
print(f'{"Jacobian top freq (Hz)":<35} {"2.77":>13} {a_freq:>13.2f}')
print(f'{"Jacobian top |lam|":<35} {"0.8670":>13} {modulus_A[idx_A[0]]:>13.4f}')
print(f'{"PSD dominant peak (Hz)":<35} {"3.12":>13} {psd_freq:>13.2f}')
print(f'{"PSD peak period (ms)":<35} {"320":>13} {1000/psd_freq:>13.0f}')
print(f'{"jPCA freq pca_dims=3 (Hz)":<35} {"---":>13} {jpca3_freq:>13.2f}')
print(f'{"jPCA freq pca_dims=6 (Hz)":<35} {"---":>13} {jpca6_freq:>13.2f}')
print("=" * 65)

## 9. Oscillation evidence summary

| Check | v1 | v2 |
|-------|-------|-------|
| W_rec: complex eigenvalues with large Im | ? | ? |
| Jacobian A: eigenvalues near unit circle | ? | ? |
| PSD: peak above 1/f baseline | ? | ? |
| PSD by CTOA: consistent frequency | ? | ? |
| jPCA M_skew: rotational frequency | N/A | ? |